# 09 · Failures and clean shutdown

**Time:** 30–45 minutes, including experiments and exercises.
**Before you start:** Review futures, Events, and asyncio tasks in lessons 02, 04, and 08.

You will make worker errors visible, separate a timeout from cancellation,
stop cooperative thread work, and let `TaskGroup` coordinate async failures.

Run cells from top to bottom in a **Python 3.11+** Jupyter kernel. Async cells use
top-level `await`: you can write `await` directly in a cell, without enclosing
it in a new async function. IPython supports this. No network, dataset, or GPU is needed.
Short sleeps explicitly simulate waiting. They do not perform CPU preprocessing.

[Course guide](README.md) · Previous: [Safe shared state](08_safe_shared_state.ipynb)
· Next: [Efficient multiprocessing](10_efficient_multiprocessing.ipynb)

## 1. Three different outcomes

| Word | Plain meaning |
| --- | --- |
| Exception | Work failed; Python carries the reason in an error object. |
| Timeout | The caller's allowed waiting time ran out. |
| Cancellation | Someone requested that work should stop, or prevented pending work from starting. |

A **future** records the result of a submitted job. It can also record the error
raised by that job. Reading the future lets the caller discover what happened.
A timeout while reading a future is about the caller's wait; the worker can still
be running. We will prove this with signals rather than relying on sleep ordering.

In [1]:
import asyncio
import sys
import threading
from concurrent.futures import CancelledError as FutureCancelledError
from concurrent.futures import ThreadPoolExecutor, as_completed
from io import StringIO

assert sys.version_info >= (3, 11), "TaskGroup and asyncio.timeout need Python 3.11+."
WAIT_SECONDS = 5  # Time limit for coordination in these small experiments.
print("Python:", sys.version.split()[0])

Python: 3.13.7


## 2. Inspect every future

Here an invalid sample fails while two valid samples succeed. Each future keeps
its own outcome. `result()` raises the error in the caller when that job failed.
`as_completed()` yields futures as they finish, so one slow job does not stop us
from collecting faster jobs. Its `timeout` limits the whole collection wait,
measured from the call to `as_completed()`; it is not a new timeout for each job.
[Future documentation](https://docs.python.org/3.11/library/concurrent.futures.html#future-objects)

**Predict:** Does one failed thread-pool job automatically cancel the other jobs?

In [2]:
def validate_sample(sample_id):
    if sample_id == "broken":
        raise ValueError("Sample has an invalid label")
    return sample_id, len(sample_id)

successes = {}
failures = {}
with ThreadPoolExecutor(max_workers=2) as pool:
    future_to_id = {
        pool.submit(validate_sample, sample_id): sample_id
        for sample_id in ("cat", "broken", "bird")
    }
    for future in as_completed(future_to_id, timeout=WAIT_SECONDS):
        sample_id = future_to_id[future]
        try:
            name, size = future.result()
            successes[name] = size
        except ValueError as error:
            failures[sample_id] = str(error)

print("Successful:", successes)
print("Failed:", failures)
assert successes == {"cat": 3, "bird": 4}
assert failures == {"broken": "Sample has an invalid label"}

Successful: {'cat': 3, 'bird': 4}
Failed: {'broken': 'Sample has an invalid label'}


**Explain:** The other jobs continue. The caller must choose whether to skip an
invalid sample, stop the pipeline, or retry an operation. Catch a specific expected
error; otherwise a programming mistake can be silently treated as a bad input.

A failed future's error does not become visible merely because we submitted it.
Keep track of all submitted futures and inspect their outcomes. Retrying is useful
only when the cause can be temporary and repeating the operation is acceptable.
Repeating an invalid label does not repair it.

## 3. A timeout does not stop a running thread

A worker signals that it has started, then waits for permission to finish.
The caller tries `result(timeout=0.02)` before giving that permission.

The worker's Event wait also has a longer limit, so this demonstration can fail
with an error rather than wait forever if we accidentally forget to release it.

**Predict:** After the caller's timeout, is the job done? Can `cancel()` stop it now?

In [3]:
worker_started = threading.Event()
may_finish = threading.Event()

def wait_for_permission():
    worker_started.set()
    if not may_finish.wait(timeout=WAIT_SECONDS):
        raise TimeoutError("Worker was not released")
    return "worker finished after release"

with ThreadPoolExecutor(max_workers=1) as pool:
    running_future = pool.submit(wait_for_permission)
    try:
        if not worker_started.wait(timeout=WAIT_SECONDS):
            raise TimeoutError("Worker did not start")
        try:
            running_future.result(timeout=0.02)
        except TimeoutError:
            print("Caller stopped waiting; worker still running:", running_future.running())
        print("Cancel running job:", running_future.cancel())
        assert running_future.running()
    finally:
        may_finish.set()  # Always let this teaching worker end.
    print(running_future.result(timeout=WAIT_SECONDS))

Caller stopped waiting; worker still running: True
Cancel running job: False
worker finished after release


**Explain:** The caller's timeout expires, but the worker is still waiting.
`cancel()` returns `False` because the job has started. Releasing the Event lets
it finish. There is no general thread-pool operation that forcibly stops one
running Python thread safely.

`ProcessPoolExecutor` futures follow the same rule: `result(timeout=...)` does not
kill the process or interrupt its job. A process-pool future can be committed to
execution before its function visibly begins, so cancellation can already fail.
Process boundaries alone do not provide a cancellation strategy.
[Future cancellation documentation](https://docs.python.org/3.11/library/concurrent.futures.html#concurrent.futures.Future.cancel)

## 4. Pending jobs can be cancelled

With one worker occupied, a second submitted job must remain pending. We can
cancel that second job before releasing the first worker. Its `result()` then
raises `concurrent.futures.CancelledError`.

**Predict:** Which `cancel()` call returns `True`: the running job or the pending job?

In [4]:
occupied = threading.Event()
release_worker = threading.Event()
pending_job_ran = threading.Event()

def hold_worker():
    occupied.set()
    if not release_worker.wait(timeout=WAIT_SECONDS):
        raise TimeoutError("Occupied worker was not released")
    return "running job completed"

def pending_job():
    pending_job_ran.set()
    return "this job should be cancelled"

with ThreadPoolExecutor(max_workers=1) as pool:
    first = pool.submit(hold_worker)
    try:
        if not occupied.wait(timeout=WAIT_SECONDS):
            raise TimeoutError("First job did not start")
        second = pool.submit(pending_job)
        print("Cancel running:", first.cancel())
        cancelled_pending = second.cancel()
        print("Cancel pending:", cancelled_pending)
        assert cancelled_pending
        try:
            second.result(timeout=WAIT_SECONDS)
        except FutureCancelledError:
            print("Pending future reports cancellation")
    finally:
        release_worker.set()
    print(first.result(timeout=WAIT_SECONDS))

assert not pending_job_ran.is_set()

Cancel running: False
Cancel pending: True
Pending future reports cancellation
running job completed


**Explain:** Cancellation prevents the pending job from starting. It leaves the
running job alone. Cancellation races with starting work in ordinary programs;
always check its return value. Our Event makes the pending state predictable.

## 5. Executor shutdown has its own meaning

| Call or pattern | What it does |
| --- | --- |
| Exit `with ThreadPoolExecutor(...)` | Waits for submitted work to finish. |
| `shutdown(wait=True, cancel_futures=True)` | Cancels jobs still pending and waits for running jobs. |
| `shutdown(wait=False, cancel_futures=True)` | Returns without waiting; running jobs continue. |

`wait=False` does not make Python exit while thread-pool work is still running.
Also, a short `result()` timeout inside a `with` block does **not** bound the time
spent leaving that block. Shutdown has no built-in timeout in these executor APIs.
Design the running jobs to finish or cooperate with a stop signal.
[Executor shutdown documentation](https://docs.python.org/3.11/library/concurrent.futures.html#concurrent.futures.Executor.shutdown)

## 6. Ask a worker to stop between units of work

**Cooperative cancellation** means a worker checks a signal and chooses to stop
at a safe point. An Event can carry that stop request. `Event.wait(0.02)` also
wakes early when the Event is set, unlike an ordinary sleep.

This worker writes tiny records to an in-memory text stream. A short wait stands
in for waiting between units. `finally` closes the stream whether work finishes,
stops, or raises an error. A finite loop is a fallback if no stop request arrives.
`StringIO` behaves like a small text file stored in memory, so this example can
practice resource cleanup without creating a file on disk.

**Predict:** Will the stop request skip the `finally` block?

In [5]:
stop_requested = threading.Event()
first_unit_done = threading.Event()
cleanup_done = threading.Event()

def cooperative_worker():
    resource = StringIO()
    completed = []
    try:
        for unit_id in range(100):
            if stop_requested.wait(timeout=0.02):
                break
            resource.write(f"processed {unit_id}\n")
            completed.append(unit_id)
            first_unit_done.set()
        return completed
    finally:
        resource.close()
        cleanup_done.set()

with ThreadPoolExecutor(max_workers=1) as pool:
    future = pool.submit(cooperative_worker)
    try:
        if not first_unit_done.wait(timeout=WAIT_SECONDS):
            raise TimeoutError("Worker did not complete its first unit")
    finally:
        stop_requested.set()
    completed_units = future.result(timeout=WAIT_SECONDS)

print("Units completed before stopping:", len(completed_units))
print("Cleanup completed:", cleanup_done.is_set())
assert 1 <= len(completed_units) <= 100
assert completed_units == list(range(len(completed_units)))
assert cleanup_done.is_set()

Units completed before stopping: 1
Cleanup completed: True


**Explain:** The worker observes the request at a checkpoint, leaves its loop,
and still runs `finally`. The exact count can vary with scheduling. We test
cleanup and result consistency, not an exact stopping time.

Checking once between huge tasks can give a slow response. Smaller units give
more stopping opportunities, with some coordination overhead. A blocking network
call also needs its own timeout; an Event cannot interrupt that call for you.

With manually managed threads, use `thread.join(timeout=...)`, then check
`thread.is_alive()`. `join()` always returns `None`; it does not tell you whether
the wait timed out. A still-alive thread has not finished. The time limit ends
the caller's wait and does not stop the thread, just as with a timed-out future.
[Thread join documentation](https://docs.python.org/3.11/library/threading.html#threading.Thread.join)

## 7. Keep related async tasks in a TaskGroup

**Structured concurrency** means keeping related tasks inside one code block
that waits for them and handles their failures together. `asyncio.TaskGroup` provides that
scope. If a child raises an error other than cancellation, the group asks its
siblings to cancel and waits for their cleanup before reporting the errors.
`async with` enters and leaves this block using async operations: leaving it
can `await` the tasks and their cleanup.

An `ExceptionGroup` is one container for one or more errors. `except* ValueError`
handles the matching errors inside that container. It can leave other error types
unhandled so they remain visible. The `*` is part of the exception-handling
syntax; it is not multiplication. We use it because the raised object groups errors.
[TaskGroup documentation](https://docs.python.org/3.11/library/asyncio-task.html#task-groups)

**Predict:** When one child fails, does its sibling's `finally` block run before
we receive the group's error?

In [6]:
async def demonstrate_group_failure():
    sibling_started = asyncio.Event()
    never_released = asyncio.Event()
    events = []
    handled_errors = []

    async def sibling():
        events.append("sibling started")
        sibling_started.set()
        try:
            await never_released.wait()
        except asyncio.CancelledError:
            events.append("sibling received cancellation")
            raise  # Let the group see that cancellation happened.
        finally:
            events.append("sibling cleaned up")

    async def failing_child():
        await sibling_started.wait()
        raise ValueError("Invalid batch shape")

    try:
        async with asyncio.timeout(WAIT_SECONDS):  # Backup limit for this experiment.
            async with asyncio.TaskGroup() as group:
                sibling_task = group.create_task(sibling())
                failed_task = group.create_task(failing_child())
    except* ValueError as errors:
        handled_errors = [str(error) for error in errors.exceptions]
        events.append("caller handled grouped error")

    return events, handled_errors, sibling_task.cancelled(), type(failed_task.exception()).__name__

group_events, group_errors, sibling_cancelled, failed_error_type = await demonstrate_group_failure()
print("\n".join(group_events))
print("Handled:", group_errors)
print("Sibling cancelled:", sibling_cancelled, "Failure type:", failed_error_type)
assert sibling_cancelled and failed_error_type == "ValueError"
assert group_errors == ["Invalid batch shape"]
assert group_events.index("sibling cleaned up") < group_events.index("caller handled grouped error")

sibling started
sibling received cancellation
sibling cleaned up
caller handled grouped error
Handled: ['Invalid batch shape']
Sibling cancelled: True Failure type: ValueError


**Explain:** The sibling has definitely started before the failure because the
failing child waits on its Event. The group cancels the waiting sibling, lets it
clean up, and then reports the `ValueError` to the caller as a grouped error.

More than one child can fail before cancellation takes effect, so groups may
contain several errors. Cancellation of a sibling is handled differently from
an ordinary child failure; it is not another `ValueError` to catch here.

`asyncio.CancelledError` and `concurrent.futures.CancelledError` are different
classes. Use the one for the concurrency mechanism you are handling.

## 8. Put an async operation inside a time limit

`asyncio.timeout(seconds)` limits the work inside its code block by cancelling the current task
when the time limit expires. At the outside of that context, the timeout becomes
a `TimeoutError`. Catch it **outside** the `async with` block.

A **coroutine** is work created by calling an `async def` function. When running,
it can receive cancellation
while it is waiting at an `await`.
Cleanup belongs in `finally`. If you catch `asyncio.CancelledError` to log or clean
up, normally re-raise it so cancellation can finish.
[Async timeout documentation](https://docs.python.org/3.11/library/asyncio-task.html#timeouts)

**Predict:** Will we receive data, a timeout, or both from the next simulated request?

In [7]:
timeout_events = []

async def simulated_request():
    try:
        await asyncio.sleep(0.15)  # Simulated I/O wait; no request is sent.
        return "sample data"
    except asyncio.CancelledError:
        timeout_events.append("request received cancellation")
        raise
    finally:
        timeout_events.append("request cleaned up")

received_data = None
timed_out = False
try:
    async with asyncio.timeout(0.02):
        received_data = await simulated_request()
except TimeoutError:
    timed_out = True
    timeout_events.append("caller received TimeoutError")

print("\n".join(timeout_events))
print("Data:", received_data)
assert timed_out and received_data is None
assert timeout_events == [
    "request received cancellation", "request cleaned up", "caller received TimeoutError"
]

request received cancellation
request cleaned up
caller received TimeoutError
Data: None


**Explain:** We receive the timeout after the coroutine's cleanup. This is different
from timing out on a thread future: here the coroutine receives cancellation.

Async cancellation is still cooperative. The **event loop** schedules async
tasks and resumes them when they can continue. Blocking it with a long
Python loop prevents timely delivery. Suppressing `CancelledError` can also stop
`TaskGroup` and timeout scopes from behaving as intended. Slow cleanup can make
observed elapsed time exceed the requested timeout.

If an async task starts a thread with `asyncio.to_thread()`, cancelling the async
wait does not forcibly stop the underlying thread. Give that thread its own stop
strategy, just as in section 6.
[Async cancellation documentation](https://docs.python.org/3.11/library/asyncio-task.html#task-cancellation)

## 9. Plan shutdown before starting the pipeline

For an ML pipeline that fails on a batch, a useful order is:

1. Stop accepting or submitting new batches.
2. Cancel futures that are still pending; inspect whether cancellation succeeded.
3. Signal running cooperative workers to stop at safe checkpoints.
4. Give blocking operations their own time limits, and release resources in `finally`.
5. Wait within an application time budget and report workers that do not finish.

In a bounded queue, a producer stuck on `put()` and a consumer stuck on `get()`
also need a way to notice shutdown. Use time-limited queue operations with stop
checks, or a carefully designed completion message. Never assume all expected
messages will arrive after a producer fails.

A time budget detects unresponsive work; it cannot guarantee that every thread
can be killed. Every runnable worker in this notebook has finite work or a
bounded wait, and its release signal is set in `finally` where needed.

## 10. Carry the idea across concurrency mechanisms

| Mechanism | How to think about stopping |
| --- | --- |
| Thread pool | Cancel pending futures; running jobs need cooperative checks and bounded blocking calls. |
| Process pool | Future timeouts do not stop jobs; plan process-safe stop signaling and cleanup. |
| Asyncio | Cancel related tasks and await their cleanup, often through a TaskGroup. |

A `threading.Event` lives in one process and is not a stop signal shared by spawned
process workers. Multiprocessing needs an appropriate process-safe signaling
arrangement. A **guarded helper script** starts workers under
`if __name__ == "__main__":`, so importing it in a new process does not launch
more workers. We use these scripts elsewhere in
the course so it works on Windows as well as other systems.

Forcefully terminating a process is a different operation from cancelling a
future. It can skip cleanup and leave queues or locks damaged. Do not treat it
as the ordinary shutdown path for a cooperative pipeline.
[Multiprocessing termination documentation](https://docs.python.org/3.11/library/multiprocessing.html#multiprocessing.Process.terminate)

## Exercises

1. Change the invalid-sample example so two samples fail. Keep successful results
   and a separate error entry for each failed input. Verify their identifiers.
2. In the pending-job experiment, move the call to `second.cancel()` until after
   the first worker is released. Why might cancellation now return either result?
   Explain why a sleep would not prove which state the future is in.
3. Add `raise ValueError("unit failed")` after the first unit in the cooperative
   worker. Catch that specific error when collecting the future. Does cleanup still run?
4. Adapt the TaskGroup experiment so the child raises `RuntimeError`, and handle
   it with `except* RuntimeError`. Verify that sibling cleanup still finishes first.
5. A developer catches an async timeout and immediately assumes a thread launched
   by `asyncio.to_thread` is gone. Explain the mistake and suggest a stop design.

In [8]:
# Your work. Re-create Events and resources before each run.
# Keep finite waits and release signals in finally when editing experiments.
exercise_inputs = ["cat", "broken_a", "bird", "broken_b"]

<details>
<summary>Worked answers — open after trying</summary>

**1.** Validate both names, collect each future, and store errors by input:

```python
def check_input(name):
    if name.startswith("broken"):
        raise ValueError(f"Invalid sample: {name}")
    return len(name)

successes, failures = {}, {}
with ThreadPoolExecutor(max_workers=2) as pool:
    submitted = {pool.submit(check_input, name): name for name in exercise_inputs}
    for future in as_completed(submitted, timeout=WAIT_SECONDS):
        name = submitted[future]
        try:
            successes[name] = future.result()
        except ValueError as error:
            failures[name] = str(error)
assert set(successes) == {"cat", "bird"}
assert set(failures) == {"broken_a", "broken_b"}
```

**2.** Once released, the worker may take the second job before the caller cancels
it. `True` means cancellation succeeded before execution was committed; `False`
means it was already running or finished. Scheduling is variable. Use signals to
stage a particular state and inspect the future rather than guessing from a sleep.

**3.** `finally` runs when the worker raises. The future stores the `ValueError`,
and `future.result()` raises it in the caller. Catch it there, then verify
`cleanup_done.is_set()`. Do not catch the error inside the worker and return a
normal-looking result unless that is your explicit failure policy.

**4.** Change the raised type and the `except*` type together, then update
`failed_error_type` and the expected error message in the checks. The sibling
cleanup order remains the same; TaskGroup reacts to either ordinary error type.

**5.** Cancellation ends the async waiting path but cannot forcibly interrupt
the running thread. Pass a thread stop Event to cooperative work, check it between
bounded units, set it in the caller's `finally`, and track completion and cleanup.

</details>

## Check your understanding

- Why does a future's timeout leave its running worker alive?
- What makes cancelling a pending job different from stopping a running job?
- Why can leaving a thread-pool `with` block take longer than a `result()` timeout?
- What must a cooperative worker do to notice a stop request?
- What does TaskGroup wait for before reporting a child error?
- Why should a coroutine normally re-raise `asyncio.CancelledError` after cleanup?

The next lesson applies these ideas to the cost of moving and dividing work
across process boundaries.

**Official references:** [Futures and executor shutdown](https://docs.python.org/3.11/library/concurrent.futures.html),
[thread Events and joins](https://docs.python.org/3.11/library/threading.html),
[TaskGroup](https://docs.python.org/3.11/library/asyncio-task.html#task-groups),
[async timeouts](https://docs.python.org/3.11/library/asyncio-task.html#timeouts),
and [IPython top-level await](https://ipython.readthedocs.io/en/stable/interactive/autoawait.html).

[Course guide](README.md) · Previous: [Safe shared state](08_safe_shared_state.ipynb)
· Next: [Efficient multiprocessing](10_efficient_multiprocessing.ipynb)